In [ ]:
#use "../lib/nb.ml";;
run_notebook "Types.ipynb"

# Minimizing a Finite State Machine

In this notebook we show how a <span style="font-variant:small-caps;">Dfa</span> can be minimized by identifying states that are equivalent.

## Some Type Definitions

* Characters have the OCaml type `char`.
* The type parameter `'q` is the type of the states that make up the <span style="font-variant:small-caps;">Dfa</span>.

`'q trans_rel_det` is the type that describes the transition function $\delta$.  It is defined in the notebook `Types.ipynb`.

In [ ]:
#show_type trans_rel_det

`'q dfa` is the type that describes a <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
#show_type dfa

`'q pair` is a pair of states.

In [ ]:
type 'q pair = 'q * 'q

The states of the minimized <span style="font-variant:small-caps;">Dfa</span> will be  sets of states of the original <span style="font-variant:small-caps;">Dfa</span>.  These states will contain those states that are *equivalent*.

In [ ]:
type 'q set_state = 'q FSet.t

In [ ]:
type 'q trans_rel_set = 'q set_state trans_rel_det

In [ ]:
type 'q min_dfa = 'q set_state dfa

The function `arb m` takes a non-empty set `m` as its argument and returns an *arbitrary* element from this set.  The set `m` is not changed.

In Python, we have to define an abstract type `S` that serves as a *type parameter* in order to type-check this
function.  In OCaml, the type variable `'s` does not have to be declared.  `FSet.choose` returns the smallest
element of the set.

**Function `arb m`**
- *Input:* `m` is a non-empty set.
- *Output:* An arbitrary element of `m`.

In [ ]:
let arb (m : 's FSet.t) : 's =
  FSet.choose m

The function `cart_prod a b` computes the *Cartesian product* $A \times B$ of the sets $A$ and $B$ where $A \times B$ is defined as follows:
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$
The type variable `'t` denotes the type of the elements of the second set.  The set comprehension of Python is
written using the operators `let*` and `return` of the module `FSet`.

**Function `cart_prod a b`**
- *Input:* `a` and `b` are sets.
- *Output:* The Cartesian product $A \times B$.

In [ ]:
let cart_prod (a : 's FSet.t) (b : 't FSet.t) : ('s * 't) FSet.t =
  let open FSet in
  let* x = a in
  let* y = b in
  return (x, y)

In [ ]:
cart_prod (FSet.of_list [ 1; 2 ]) (FSet.of_list [ 'a'; 'b'; 'c' ])

The function `separate` takes four arguments:
- `pairs`  is a set of pairs of states from some given <span style="font-variant:small-caps;">Dfa</span> $F$.

   If $(p_1, p_2) \in \texttt{pairs}$, then $p_1$ and $p_2$ are known to be *separable*.
   
   Two states $p_1$ and $p_2$ are *separable* if there is a string $w$ such that processing $w$ in state
   $p_1$ leads to an accepting state, while processing $w$ in state $p_2$ leads to a state that is not 
   accepting, or vice versa.
- `states` is the set of all states of the <span style="font-variant:small-caps;">Dfa</span> $F$,
- `sigma` is the alphabet of the <span style="font-variant:small-caps;">Dfa</span> $F$.
- `delta` is the transition function of the <span style="font-variant:small-caps;">Dfa</span>

The function `separate pairs states sigma delta` computes the set of pairs of states $(q_1, q_2)$ that are *separable* because there is some character $c \in \Sigma$ such that 
$$\delta(q_1,c) = p_1, \quad \textrm{but} \quad \delta(q_2,c) = p_2 $$
and $(p_1, p_2)$ is already known to be *separable* because $(p_1, p_2) \in \textrm{pairs}$.

In the set comprehension, the condition `if ... in Pairs` of Python becomes `if ... then return (q1, q2) else empty`.

**Function `separate pairs states sigma delta`**
- *Input:* `pairs` is a set of pairs of states that are known to be separable, `states` is the set of states of a <span style="font-variant:small-caps;">Dfa</span>, `sigma` is its alphabet, and `delta` is its transition function.
- *Output:* The set of all pairs $(q_1, q_2)$ such that $(\delta(q_1, c), \delta(q_2, c)) \in \texttt{Pairs}$ for some character $c$.

In [ ]:
let separate (pairs : 'q pair FSet.t) (states : 'q FSet.t) (sigma : char FSet.t) (delta : 'q trans_rel_det)
    : 'q pair FSet.t =
  let open FSet in
  let* q1 = states in
  let* q2 = states in
  let* c  = sigma in
  if mem (Dict.find (q1, c) delta, Dict.find (q2, c) delta) pairs then return (q1, q2) else empty

Given a state `p` and a `partition` of the set of all states, the function `find_equivalence_class p partition` returns the equivalence class of `p`, i.e. it returns the set of states from `partition` that contains the state `p`. 

**Function `find_equivalence_class p partition`**
- *Input:* `p` is a state and `partition` is a partition of the set of states.
- *Output:* The set in `partition` that contains `p`.

In [ ]:
let find_equivalence_class (p : 'q) (partition : 'q FSet.t FSet.t) : 'q FSet.t =
  arb (FSet.filter (fun c -> FSet.mem p c) partition)

The function `all_separable q a sigma delta` takes four arguments:
* `q` is the set of states of the <span style="font-variant:small-caps;">Dfa</span>.
* `a` is the set of all accepting states,
* `sigma`  is the alphabet.
* `delta` is the transition function.  

  `delta` is represented as a dictionary.   

The function computes the set of all Pairs `(p, q)` such that `p` and `q` are separable, i.e. all pairs such that
$$ \exists s \in \Sigma^*: \bigl(\delta^*(p, s) \in A \wedge \delta^*(q,s) \not\in A\bigr) \vee 
                           \bigl(\delta^*(p, s) \not\in A \wedge \delta^*(q,s) \in A\bigr). 
$$
The result is computed via a *fixed point* computation:
1. Initially, all accepting states are separated from the non-accepting states.
2. If $p_1$ and $p_2$ are separable and $q_1$ and $q_2$ are two states such that there is a character 
   $c \in \Sigma$ such that
   $$ \delta(q_1, c) = p_1 \quad \mbox{and} \quad \delta(q_2, c) = p_2, $$
   then $q_1$ and $q_2$ are separable.

**Function `all_separable q a sigma delta`**
- *Input:* `q` is the set of states of a complete <span style="font-variant:small-caps;">Dfa</span>, `a` is its set of accepting states, `sigma` is its alphabet, and `delta` is its transition function.
- *Output:* The set of all pairs of separable states.

In [ ]:
let all_separable (q : 'q FSet.t) (a : 'q FSet.t) (sigma : char FSet.t) (delta : 'q trans_rel_det) : 'q pair FSet.t =
  let rec loop (separable : 'q pair FSet.t) : 'q pair FSet.t =
    let newPairs = separate separable q sigma delta in
    if FSet.subset newPairs separable then separable
    else loop (FSet.union separable newPairs)
  in
  loop (FSet.union (cart_prod (FSet.diff q a) a) (cart_prod a (FSet.diff q a)))

The function `reachable q0 sigma delta` takes three arguments:
* `q0` is the start state of a <span style="font-variant:small-caps;">Dfa</span>,
* `sigma`  is the alphabet.
* `delta`  is the transition function.  The transition function is assumed to be *complete*. `delta` is represented as a dictionary.   

It returns the set of all states that can be reached from the start state `q0` by reading strings of characters from `sigma`.

The result is computed via a *fixed point* computation.

**Function `reachable q0 sigma delta`**
- *Input:* `q0` is the start state of a complete <span style="font-variant:small-caps;">Dfa</span>, `sigma` is its alphabet, and `delta` is its transition function.
- *Output:* The set of all states that can be reached from `q0`.

In [ ]:
let reachable (q0 : 'q) (sigma : char FSet.t) (delta : 'q trans_rel_det) : 'q FSet.t =
  let rec loop (result : 'q FSet.t) : 'q FSet.t =
    let newStates =
      let open FSet in
      let* p = result in
      let* c = sigma in
      return (Dict.find (p, c) delta)
    in
    if FSet.subset newStates result then result
    else loop (FSet.union result newStates)
  in
  loop (FSet.singleton q0)

The function `minimize f` takes a <span style="font-variant:small-caps;">Dfa</span> `f` as its input.
Here `f` is a record that represents the 5-tuple
$$ F = (Q, \Sigma, \delta, q_0, A). $$
The <span style="font-variant:small-caps;">Dfa</span> $F$ has to be *complete*, i.e. $\delta(q, c)$ has to be defined for all states $q$ and all
characters $c$.
The algorithm performs the following steps:
1. All unreachable states are eliminated.
2. All accepting states are separated from all non-accepting states.
3. States are separated as long as possible.
   Two states $p_1$ and $p_2$ are separable if there is a string 
   $w \in \Sigma^*$ such that 
   $$ \Bigl(\delta^*(p_1,w) \in A \;\wedge\; \delta^*(p_2,w) \not\in A\Bigr) \quad \vee \quad 
      \Bigl(\delta^*(p_1,w) \not\in A \;\wedge\; \delta^*(p_2,w) \in A\Bigr)
   $$
4. States that are not separable are *equivalent* and are therefore identified and grouped
   in equivalence classes.  The states in an equivalence class are then identified.

As in the Python version, the new transition function is built in a loop.  Since dictionaries are immutable,
`newDelta` is a reference to a dictionary, and every iteration replaces this dictionary by an extended one.

**Function `minimize f`**
- *Input:* `f` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> with the minimal number of states that accepts the same language as `f`.  Its states are sets of equivalent states of `f`.

In [ ]:
let minimize (f : 'q dfa) : 'q min_dfa =
  let ({ sigma; delta; start = q0; accepting = a; _ } : 'q dfa) = f in
  let q            = reachable q0 sigma delta in
  let separable    = all_separable q a sigma delta in
  let equivalent   = FSet.diff (cart_prod q q) separable in
  let equivClasses = FSet.map (fun q' -> FSet.filter (fun p -> FSet.mem (p, q') equivalent) q) q in
  let newQ0        = arb (FSet.filter (fun m -> FSet.mem q0 m) equivClasses) in
  let newAccept    = FSet.filter (fun m -> FSet.mem (arb m) a) equivClasses in
  let newDelta : 'q trans_rel_set ref = ref Dict.empty in
  FSet.iter
    (fun q' ->
      let classOfQ = find_equivalence_class q' equivClasses in
      FSet.iter
        (fun c -> newDelta := Dict.add (classOfQ, c) (find_equivalence_class (Dict.find (q', c) delta) equivClasses) !newDelta)
        sigma)
    q;
  { states = equivClasses; sigma; delta = !newDelta; start = newQ0; accepting = newAccept }